# 52. 원판 입력 640과 512에서 오버레이 시간과 궤적 비교

자세 모델은 640으로 고정합니다. 검증 영상 c04와 b02를 한 번씩 읽으면서 원판과 자세를 그리고 원본 크기로 저장합니다. 아래 표의 시간에는 Python 실행, 모델 준비, 추론, 영상 저장과 종료가 포함됩니다. 모델별 전처리, 원판 연결 반경, 가중치는 두 조건에서 같습니다.

In [ ]:
import pandas as pd
from IPython.display import display
from wlcv.annotation.hub_points import hub_editor
from wlcv.evaluation.plate_size_mps import benchmark, PARENT, B02
from wlcv.evaluation.plate_size_metrics import LIFT_HUB, evaluate, plot_comparison

### 선택 판독 · b02 이륙 20번 원본

20번 허브의 수동 위치가 있으면 20→30번 수동 상승량도 계산합니다. 예측을 보기 전에 원본에서 허브 중심을 한 번 누르거나 ‘판독 불가로 저장’을 선택하세요. 이 셀을 건너뛰어도 모델끼리 비교할 수 있습니다. 이후 표에 `NaN`이 보이면 해당 수동점이나 모델 출력이 비어 있다는 뜻입니다.

In [ ]:
display(hub_editor(20, path=LIFT_HUB, frames=(20,), video=B02))

### 측정표 열기

기존 측정을 먼저 엽니다. Mac에서 다시 측정하려면 `run_again = True`로 바꿉니다. 새 프로세스에서 각 조건을 네 번씩 실행하고 출력은 모두 로컬에 저장합니다.

In [ ]:
run_again = False
if run_again:
    folder = benchmark()
else:
    for table in sorted(PARENT.glob("*/runs.csv"), reverse=True):
        if table.stat().st_size and len(pd.read_csv(table)) == 16:
            folder = table.parent
            break
    else:
        raise FileNotFoundError("완료된 16회 측정표가 없습니다. run_again을 True로 바꿔 실행하세요.")
runs, motion, points, rises = evaluate(folder)
display(runs.pivot_table(index=["video_id", "size"], columns="block", values="command_s").round(2))
display(runs.groupby(["video_id", "size"])[["command_s", "pipeline_s"]].median().round(2))

### 동작 중 미탐

직접·시간 연결·미탐은 프로그램의 최종 출력 상태입니다. 정답 박스가 있는 프레임에서만 IoU 0.50을 적용합니다. 출력이 없거나 IoU가 0.50 미만이면 정답 원판을 놓친 것으로 셉니다. c04는 ready~finish, b02 하프 동작은 ready~접촉까지 포함합니다.

In [ ]:
display(pd.crosstab([motion.video_id, motion["size"]], motion.plate_status))
summary = motion.groupby(["video_id", "size"]).match.agg(구간장수="size", 정답박스="count", 정탐="sum")
summary["미탐"] = summary["정답박스"] - summary["정탐"]
display(summary)
missed = motion[motion.match.eq(False)]
display(missed[["video_id", "size", "frame_index", "plate_status", "iou"]])

### b02 허브 세로 오차와 이륙→무릎 통과 상승량

허브 세로 오차는 모델 박스 중심 y에서 수동 허브 y를 뺍니다. 양수는 화면 아래입니다. 상승량은 이륙 y에서 무릎 통과 y를 빼며, 양수가 화면 위 방향입니다. 30~36번은 최초 연속 판독만 사용합니다.

In [ ]:
display(points.pivot(index="frame_index", columns="size", values="dy_px").round(2))
paired = points.pivot(index="frame_index", columns="size", values="abs_dy_px").dropna()
display(paired.agg(["count", "mean", "median"]).round(2))
display(rises.round(2))
figure = plot_comparison(runs, points)
figure.savefig(folder / "size_comparison.png", dpi=150)
display(figure)